# Correzione della distorsione a barile con Discorpy

Questo notebook ricostruisce l'intera pipeline che abbiamo discusso:

1. **Estrazione dei punti** di una griglia di calibrazione (scacchiera) disegnata
   con puntini rossi su due immagini separate: `h.png` (linee orizzontali) e
   `v.png` (linee verticali). Funziona anche con griglie "a puntini" sparsi
   (dot-grid), oltre che con linee continue.
2. **Fit di verifica** (parabole di 2 grado) - solo per controllare
   visivamente la qualita dei punti estratti, *non* e il fit usato da Discorpy.
3. **Stima della distorsione con Discorpy**: centro di distorsione +
   coefficienti del modello polinomiale (Discorpy fa il proprio fit
   internamente, partendo dai punti grezzi che gli passiamo).
4. **Correzione di un'immagine reale** (es. una foto della stessa scacchiera)
   usando il modello stimato.

> **Nota importante**: Discorpy riceve sempre i *punti grezzi* estratti dalle
> immagini, non le parabole che fittiamo noi per la verifica visiva. Il fit
> "ufficiale" usato per calcolare il modello di distorsione lo fa Discorpy
> stesso, internamente, con le sue funzioni `find_cod_coarse`/`find_cod_fine`/
> `calc_coef_backward`.


## 0. Setup: import e installazione di Discorpy

In [ ]:
import sys
!{sys.executable} -m pip install --user discorpy

In [ ]:
# Se discorpy non e' gia' installato, decommenta la riga seguente
import sys
!{sys.executable} -m pip install discorpy --quiet --user

import os
import csv

import numpy as np
import cv2
import matplotlib.pyplot as plt

import discorpy.proc.processing as proc
import discorpy.post.postprocessing as post

print('OpenCV:', cv2.__version__)
import discorpy
print('Discorpy importato correttamente')

## 1. Parametri

Modifica questi path/parametri in base ai tuoi file.

- `H_IMAGE_PATH` / `V_IMAGE_PATH`: le due immagini con le linee rosse
  (orizzontali e verticali) della griglia di calibrazione. Se la tua griglia
  e di tipo "dots" con H e V nella stessa immagine, imposta entrambi i path
  allo stesso file.
- `TARGET_IMAGE_PATH`: l'immagine reale da correggere (es. una foto della
  scacchiera, o qualunque altra foto scattata con la stessa fotocamera/lente).
- `N_H` / `N_V`: numero di linee orizzontali / verticali nella griglia.
- `GRID_MODE`: `"lines"` se le linee sono continue/tratteggiate dense (si puo
  fare clustering riga-per-riga), `"dots"` se sono puntini sparsi radi (serve
  blob-detection + classificazione).


In [ ]:
H_IMAGE_PATH = 'chessbao.png'      # qui usiamo la dot-grid come esempio
V_IMAGE_PATH = 'chessbao.png'      # stessa immagine: contiene sia H che V
TARGET_IMAGE_PATH = '/private/camera/14334/im143340.npy'     # immagine reale da correggere

N_H = 4   # numero di linee orizzontali
N_V = 7   # numero di linee verticali

GRID_MODE = 'dots'   # 'lines' oppure 'dots'

OUTDIR = 'output_discorpy'
os.makedirs(OUTDIR, exist_ok=True)

## 2. Estrazione dei punti dalle immagini

Due modalita, a seconda di come e disegnata la griglia:

- **`lines`**: per ogni colonna (riga) dell'immagine, raggruppiamo i pixel
  rossi contigui in "blob" e prendiamo il centro di ciascun gruppo. Le linee
  orizzontali non si incrociano mai tra loro (lo stesso vale per le
  verticali), quindi possiamo assegnare i cluster trovati per **ordine**
  (dall'alto in basso / da sinistra a destra) senza un tracking complesso.

- **`dots`**: i puntini sono sparsi e non clusterizzabili riga-per-riga (i
  gap tra un puntino e il successivo sulla stessa linea sono troppo ampi e
  irregolari). Qui invece: troviamo ogni puntino come blob connesso
  (`cv2.connectedComponentsWithStats`), poi classifichiamo ogni punto come
  appartenente a una linea orizzontale o verticale guardando se ha piu
  vicini a y simile (riga orizzontale) o a x simile (colonna verticale),
  infine clusterizziamo (k-means 1D) separatamente le y dei punti
  orizzontali e le x dei punti verticali.


In [ ]:
def red_mask(img, r_min=150, g_max=100, b_max=100):
    # Maschera binaria (0/1) dei pixel 'rossi' in un'immagine BGR.
    b, g, r = img[:, :, 0], img[:, :, 1], img[:, :, 2]
    mask = (r > r_min) & (g < g_max) & (b < b_max)
    return mask.astype(np.uint8)


def clusters_1d(line, gap=15):
    # Raggruppa pixel accesi in una riga/colonna 1D in blob contigui
    # (gap massimo `gap`) e ritorna il centro di ciascun gruppo, ordinati.
    pos = np.where(line > 0)[0]
    if len(pos) == 0:
        return []
    groups = []
    start = pos[0]
    prev = pos[0]
    for p in pos[1:]:
        if p - prev > gap:
            groups.append((start, prev))
            start = p
        prev = p
    groups.append((start, prev))
    return sorted((a + b) / 2.0 for a, b in groups)


def extract_points_lines_mode(h_img, v_img, n_h, n_v):
    # Estrazione per griglie con linee continue/tratteggiate dense.
    h_mask = red_mask(h_img)
    v_mask = red_mask(v_img)
    H, W = h_mask.shape

    h_xs = [[] for _ in range(n_h)]
    h_ys = [[] for _ in range(n_h)]
    for x in range(W):
        centers = clusters_1d(h_mask[:, x])
        if len(centers) == n_h:
            for i in range(n_h):
                h_xs[i].append(x)
                h_ys[i].append(centers[i])

    v_xs = [[] for _ in range(n_v)]
    v_ys = [[] for _ in range(n_v)]
    for y in range(H):
        centers = clusters_1d(v_mask[y, :])
        if len(centers) == n_v:
            for i in range(n_v):
                v_ys[i].append(y)
                v_xs[i].append(centers[i])

    return h_xs, h_ys, v_xs, v_ys

In [ ]:
from scipy.cluster.vq import kmeans2


def assign_clusters_1d(vals, k, seed=0):
    # K-means 1D con inizializzazione su percentili equispaziati;
    # ritorna le etichette riordinate in modo che il cluster 0 sia
    # quello col centro piu piccolo, ecc.
    vals = np.asarray(vals, dtype=float)
    init = np.percentile(vals, np.linspace(5, 95, k))
    centers, labels = kmeans2(vals, init, minit='matrix', seed=seed)
    order = np.argsort(centers)
    remap = {old: new for new, old in enumerate(order)}
    labels_sorted = np.array([remap[l] for l in labels])
    return labels_sorted, np.sort(centers)


def extract_points_dots_mode(img, n_h, n_v, tol=15.0):
    # Estrazione per griglie 'a puntini' sparsi (dot-grid).
    mask = red_mask(img)
    n_blobs, labels_im, stats, centroids = cv2.connectedComponentsWithStats(
        mask, connectivity=8
    )
    pts = centroids[1:]  # esclude lo sfondo (label 0); colonne: (x, y)

    xs, ys = pts[:, 0], pts[:, 1]
    # conta vicini con y simile (riga orizzontale) e x simile (colonna verticale)
    count_h = np.array([np.sum(np.abs(ys - y) < tol) for y in ys])
    count_v = np.array([np.sum(np.abs(xs - x) < tol) for x in xs])
    is_horizontal = count_h > count_v

    h_pts = pts[is_horizontal]
    v_pts = pts[~is_horizontal]

    h_line_labels, h_centers = assign_clusters_1d(h_pts[:, 1], n_h)
    v_line_labels, v_centers = assign_clusters_1d(v_pts[:, 0], n_v)

    h_xs = [h_pts[h_line_labels == i, 0].tolist() for i in range(n_h)]
    h_ys = [h_pts[h_line_labels == i, 1].tolist() for i in range(n_h)]
    v_xs = [v_pts[v_line_labels == i, 0].tolist() for i in range(n_v)]
    v_ys = [v_pts[v_line_labels == i, 1].tolist() for i in range(n_v)]

    return h_xs, h_ys, v_xs, v_ys

In [ ]:
# --- Esegue l'estrazione in base alla modalita scelta ---
h_img = cv2.imread(H_IMAGE_PATH)
v_img = cv2.imread(V_IMAGE_PATH)
if h_img is None:
    raise FileNotFoundError(f'Non trovo {H_IMAGE_PATH}')
if v_img is None:
    raise FileNotFoundError(f'Non trovo {V_IMAGE_PATH}')

if GRID_MODE == 'lines':
    h_xs, h_ys, v_xs, v_ys = extract_points_lines_mode(h_img, v_img, N_H, N_V)
elif GRID_MODE == 'dots':
    # nel caso 'dots' tipicamente H e V sono nella stessa immagine;
    # se sono separate basta chiamare la funzione su entrambe e unire i risultati
    h_xs1, h_ys1, v_xs1, v_ys1 = extract_points_dots_mode(h_img, N_H, N_V)
    if H_IMAGE_PATH == V_IMAGE_PATH:
        h_xs, h_ys, v_xs, v_ys = h_xs1, h_ys1, v_xs1, v_ys1
    else:
        h_xs2, h_ys2, v_xs2, v_ys2 = extract_points_dots_mode(v_img, N_H, N_V)
        h_xs, h_ys = h_xs1, h_ys1
        v_xs, v_ys = v_xs2, v_ys2
else:
    raise ValueError("GRID_MODE deve essere 'lines' o 'dots'")

print('Punti per linea orizzontale:', [len(x) for x in h_xs])
print('Punti per linea verticale:  ', [len(x) for x in v_xs])

## 3. Fit di verifica (parabole) - solo per controllo visivo

**Questo fit NON viene passato a Discorpy.** Serve solo a:
- controllare visivamente che l'estrazione dei punti sia corretta;
- avere un'indicazione rapida della qualita (RMS dei residui) prima di
  procedere con la stima vera e propria.

Usiamo un fit robusto con rigetto iterativo degli outlier.


In [ ]:
def robust_polyfit(x, y, deg=2, n_iter=6, thresh=3.0, min_std=1.0):
    # Fit polinomiale con rigetto iterativo degli outlier.
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.ones(len(x), dtype=bool)
    for _ in range(n_iter):
        coef = np.polyfit(x[mask], y[mask], deg)
        resid_all = y - np.polyval(coef, x)
        std = np.std(resid_all[mask])
        new_mask = np.abs(resid_all) < max(thresh * std, min_std)
        if new_mask.sum() == mask.sum():
            mask = new_mask
            break
        mask = new_mask
    coef = np.polyfit(x[mask], y[mask], deg)
    resid = y[mask] - np.polyval(coef, x[mask])
    return coef, mask, resid


h_coefs = []
print('Fit di verifica - linee orizzontali (y = a*x^2 + b*x + c):')
for i in range(N_H):
    coef, mask, resid = robust_polyfit(h_xs[i], h_ys[i])
    h_coefs.append(coef)
    rms = np.sqrt(np.mean(resid ** 2))
    print(f'  H{i}: kept {mask.sum()}/{len(mask)}  RMS={rms:.2f}px  max={np.max(np.abs(resid)):.2f}px')

v_coefs = []
print('Fit di verifica - linee verticali (x = a*y^2 + b*y + c):')
for i in range(N_V):
    coef, mask, resid = robust_polyfit(v_ys[i], v_xs[i])
    v_coefs.append(coef)
    rms = np.sqrt(np.mean(resid ** 2))
    print(f'  V{i}: kept {mask.sum()}/{len(mask)}  RMS={rms:.2f}px  max={np.max(np.abs(resid)):.2f}px')

In [ ]:
# Visualizza i punti grezzi + le parabole di verifica
fig, ax = plt.subplots(figsize=(12, 7))
h_colors = plt.cm.Reds(np.linspace(0.45, 0.9, N_H))
v_colors = plt.cm.Blues(np.linspace(0.45, 0.9, N_V))

for i in range(N_H):
    ax.scatter(h_xs[i], h_ys[i], s=6, color=h_colors[i])
    xs_plot = np.linspace(min(h_xs[i]), max(h_xs[i]), 300)
    ax.plot(xs_plot, np.polyval(h_coefs[i], xs_plot), color=h_colors[i], linewidth=1.5)

for i in range(N_V):
    ax.scatter(v_xs[i], v_ys[i], s=6, color=v_colors[i])
    ys_plot = np.linspace(min(v_ys[i]), max(v_ys[i]), 300)
    ax.plot(np.polyval(v_coefs[i], ys_plot), ys_plot, color=v_colors[i], linewidth=1.5)

ax.invert_yaxis()
ax.set_aspect('equal')
ax.set_xlabel('x (px)')
ax.set_ylabel('y (px)')
ax.set_title('Punti estratti + parabole di verifica (non usate da Discorpy)')
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, '00_verifica_punti_e_parabole.png'), dpi=130)
plt.show()

## 4. Conversione nel formato richiesto da Discorpy

Discorpy si aspetta `list_hor_lines` / `list_ver_lines`: liste di array
**Nx2 con coordinate (y, x)** - attenzione, ordine invertito rispetto al
solito (x, y) - una per ciascuna linea, con i **punti grezzi** (non i
coefficienti delle parabole calcolate sopra).


In [ ]:
def to_discorpy_format(xs_list, ys_list):
    # Converte liste di liste (x, y) in liste di array Nx2 (y, x),
    # come richiesto dalle funzioni di discorpy.proc / discorpy.post.
    out = []
    for xs, ys in zip(xs_list, ys_list):
        arr = np.stack([np.asarray(ys, dtype=float), np.asarray(xs, dtype=float)], axis=1)
        out.append(arr)
    return out


list_hor_lines = to_discorpy_format(h_xs, h_ys)
list_ver_lines = to_discorpy_format(v_xs, v_ys)

print(f'{len(list_hor_lines)} linee orizzontali, {len(list_ver_lines)} linee verticali pronte per Discorpy')

## 5. Stima del centro di distorsione

`proc.find_cod_coarse` stima il centro cercando dove il coefficiente
quadratico delle parabole (fittate **internamente da Discorpy**) cambia
segno. Con poche linee e rumore di estrazione, questo puo essere instabile
(il segno puo oscillare vicino al centro invece di cambiare una volta sola).

Per maggiore robustezza, partiamo dal **centro geometrico dell'immagine**
(ragionevole per la maggior parte delle fotocamere) e raffiniamo con
`find_cod_fine`, che fa una ricerca locale attorno alla stima iniziale.
Se preferisci, puoi comunque provare `find_cod_coarse` e confrontare.


In [ ]:
# Dimensioni dell'immagine su cui sono stati estratti i punti
H_grid, W_grid = h_img.shape[:2]

# --- Stima coarse (solo per confronto/debug) ---
xcenter_coarse, ycenter_coarse = proc.find_cod_coarse(list_hor_lines, list_ver_lines)
print(f'Stima coarse:           xcenter={xcenter_coarse:.2f}, ycenter={ycenter_coarse:.2f}')

# --- Stima di partenza piu robusta: centro geometrico dell'immagine ---
xcenter0, ycenter0 = W_grid / 2.0, H_grid / 2.0
print(f'Centro geometrico:      xcenter={xcenter0:.2f}, ycenter={ycenter0:.2f}')

# --- Raffinamento locale ---
all_pts = np.vstack(list_hor_lines + list_ver_lines)
point_dist = float(np.median(np.abs(np.diff(np.sort(all_pts[:, 1]))))) or 50.0

xcenter, ycenter = proc.find_cod_fine(
    list_hor_lines, list_ver_lines, xcenter0, ycenter0, point_dist
)
print(f'Centro raffinato:       xcenter={xcenter:.2f}, ycenter={ycenter:.2f}')

In [ ]:
# Visualizza i punti + il centro stimato
fig, ax = plt.subplots(figsize=(12, 7))
for line in list_hor_lines:
    ax.plot(line[:, 1], line[:, 0], '.', markersize=3, color='red')
for line in list_ver_lines:
    ax.plot(line[:, 1], line[:, 0], '.', markersize=3, color='blue')
ax.plot(xcenter, ycenter, 'g+', markersize=20, markeredgewidth=3,
        label=f'Centro raffinato ({xcenter:.1f}, {ycenter:.1f})')
ax.plot(xcenter_coarse, ycenter_coarse, 'mx', markersize=14, markeredgewidth=3,
        label=f'Centro coarse ({xcenter_coarse:.1f}, {ycenter_coarse:.1f})')
ax.invert_yaxis()
ax.set_aspect('equal')
ax.set_xlabel('x (px)')
ax.set_ylabel('y (px)')
ax.legend()
ax.set_title('Punti di griglia + centro di distorsione stimato')
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, '01_centro_distorsione.png'), dpi=130)
plt.show()

## 6. Stima dei coefficienti del modello di distorsione

`proc.calc_coef_backward` fitta (internamente) le linee con un polinomio e
calcola i coefficienti del **modello backward**: per ogni pixel
dell'immagine corretta, dice da dove "pescare" il valore nell'immagine
distorta originale.

`num_coef` e il numero di coefficienti del polinomio (tipicamente 4-5; piu
alto = modello piu flessibile, ma rischio di overfit se i punti sono pochi
o rumorosi).


In [ ]:
NUM_COEF = 5

list_fact = proc.calc_coef_backward(
    list_hor_lines, list_ver_lines, xcenter, ycenter, NUM_COEF
)

print('Coefficienti del modello di distorsione (backward):')
for i, c in enumerate(list_fact):
    print(f'  fact[{i}] = {c:.6e}')

## 7. Valutazione dei residui

Confrontiamo quanto sono "dritte" le linee della griglia **dopo** aver
applicato il modello di correzione ai punti stessi (non all'immagine): se il
modello e buono, il residuo (quanto le linee orizzontali si discostano
dall'essere perfettamente orizzontali, e viceversa per le verticali) deve
essere piccolo.


In [ ]:
list_uhor_line = post.unwarp_line_backward(list_hor_lines, xcenter, ycenter, list_fact)
list_uver_line = post.unwarp_line_backward(list_ver_lines, xcenter, ycenter, list_fact)

resid_hor = post.calc_residual_hor(list_uhor_line, xcenter, ycenter)
resid_ver = post.calc_residual_ver(list_uver_line, xcenter, ycenter)

rms_hor = np.sqrt(np.mean(resid_hor[:, 1] ** 2))
rms_ver = np.sqrt(np.mean(resid_ver[:, 1] ** 2))
print(f'Residuo RMS linee orizzontali dopo correzione: {rms_hor:.3f} px')
print(f'Residuo RMS linee verticali dopo correzione:   {rms_ver:.3f} px')

## 8. Applicazione della correzione a un'immagine reale

In [ ]:
target_img = np.load('/private/camera/14334/im143340.npy')
if target_img is None:
    raise FileNotFoundError(f'Non trovo {TARGET_IMAGE_PATH}')

H_target, W_target = target_img.shape[:2]
print(f'Immagine target: {W_target} x {H_target}')
print(f'Immagine su cui erano stati estratti i punti: {W_grid} x {H_grid}')

# Se l'immagine target ha una risoluzione diversa da quella usata per
# estrarre i punti della griglia, dobbiamo riscalare xcenter/ycenter (il
# modello polinomiale resta valido SOLO se lo scaling e isotropico, cioe se
# il rapporto W_target/W_grid e ~ uguale a H_target/H_grid; altrimenti
# bisognerebbe ripetere l'estrazione/fit direttamente sulla risoluzione target).
scale_x = W_target / W_grid
scale_y = H_target / H_grid
print(f'scale_x = {scale_x:.4f}, scale_y = {scale_y:.4f}')

xcenter_t = xcenter * scale_x
ycenter_t = ycenter * scale_y

if abs(scale_x - scale_y) / scale_x > 0.02:
    print('ATTENZIONE: scale_x e scale_y differiscono di oltre il 2%: '
          'il riscalamento del modello polinomiale potrebbe non essere accurato. '
          'In tal caso conviene ri-estrarre i punti direttamente sulla risoluzione target.')

In [ ]:
# Applica la correzione canale per canale (Discorpy lavora su array 2D)
channels = cv2.split(target_img)
corrected_channels = [
    post.unwarp_image_backward(ch.astype(np.float32), xcenter_t, ycenter_t, list_fact)
    for ch in channels
]
corrected = cv2.merge(corrected_channels).astype(np.uint8)

corrected_path = os.path.join(OUTDIR, 'immagine_corretta.png')
cv2.imwrite(corrected_path, corrected)
print(f'Immagine corretta salvata in: {corrected_path}')

In [ ]:
# Confronto visivo originale vs corretta
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(target_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Originale (distorta)')
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(corrected, cv2.COLOR_BGR2RGB))
axes[1].set_title('Corretta (Discorpy)')
axes[1].axis('off')
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, '02_confronto_originale_corretta.png'), dpi=130)
plt.show()

## 9. Salvataggio dei coefficienti per riutilizzo futuro

In [ ]:
coef_path = os.path.join(OUTDIR, 'discorpy_coefficients.txt')
with open(coef_path, 'w') as f:
    f.write(f'# Coefficienti stimati sull immagine {W_grid}x{H_grid}\n')
    f.write(f'xcenter = {xcenter}\n')
    f.write(f'ycenter = {ycenter}\n')
    for i, c in enumerate(list_fact):
        f.write(f'fact[{i}] = {c}\n')

print(f'Coefficienti salvati in: {coef_path}')
print()
with open(coef_path) as f:
    print(f.read())

## 10. Riapplicare il modello salvato ad altre immagini

Una volta stimati `xcenter`, `ycenter` e `list_fact`, puoi applicarli a
qualunque altra foto scattata con la stessa fotocamera/lente nelle stesse
condizioni, senza ripetere la stima. Esempio:

```python
import numpy as np
import cv2
import discorpy.post.postprocessing as post

# Ricarica i coefficienti salvati (xcenter, ycenter, list_fact)
# leggendoli da discorpy_coefficients.txt

img = cv2.imread("altra_foto.png")
channels = cv2.split(img)
corrected = cv2.merge([
    post.unwarp_image_backward(ch.astype(np.float32), xcenter, ycenter, list_fact)
    for ch in channels
]).astype("uint8")
cv2.imwrite("altra_foto_corretta.png", corrected)
```
